# 11. Colon mct mc donor

Part of the **[Fig. 6 chapter](fig6.md)** — see it for the panel-by-panel map, run order, and data inputs. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)).


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
# All absolute paths below resolve from these two roots; the working directory
# is the original analysis/ folder, and repro_guard prevents any existing file
# from being overwritten when you re-run this notebook. See the book's
# 'Reproduction guide' chapter for details.
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration
from ALLCools.mcds import MCDS

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'


In [ ]:
def dump_embedding(adata, name, n_dim=2):
    # put manifold coordinates into adata.obs
    for i in range(n_dim):
        adata.obs[f'{name}_{i}'] = adata.obsm[f'X_{name}'][:, i]
    return adata


In [ ]:
indir = f'{ENTEX_ROOT}/'
mct_dir = f'{indir}mCT/'
m3c_dir = f'{indir}mcds/'
outdir = f'{indir}analysis/colon_mct/'
tissue = 'TrCo'

In [ ]:
var_dim = 'chrom5k'
chrom_to_remove = ['chrX', 'chrY', 'chrM', 'chrL']
black_list_path = f'{REF_ROOT}/blacklist/hg38-blacklist.v2.bed.gz'


In [ ]:
meta = anndata.read_h5ad(f'{mct_dir}entex_2_rna.h5ad', 'r').obs
selc = meta.index[(meta['FinalDNAReads']>500000) & (meta['mCCCFrac']<0.1)]
print(len(selc))


In [ ]:
mcds = MCDS.open(f'{mct_dir}entex_2.mcds', use_obs=selc, var_dim=var_dim)
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds

In [ ]:
mcad = mcds.get_score_adata(mc_type='CGN', quant_type='hypo')
binarize_matrix(mcad, cutoff=0.95)
filter_regions(mcad, n_cell=5)
remove_black_list_region(mcad, black_list_path=black_list_path)
mcad.write_h5ad(f'{outdir}5kCG.h5ad')


In [ ]:
model = LSI(scale_factor=10000,
            n_components=50,
            algorithm='arpack',
            random_state=0)


In [ ]:
raw_key = '5kCG_lsi'
obsm_key = 'X_lsi'
model.fit_transform(mcad, obsm_name='5kCG_lsi')
npc = significant_pc_test(mcad, p_cutoff=0.1, obsm=raw_key, update=False)


In [ ]:
mcad.obs['Donor'] = 'PT-1LVAN'

In [ ]:
mcad.obsm[obsm_key] = normalize(mcad.obsm[raw_key][:, :npc], axis=1)
tsne(mcad, obsm=obsm_key, metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
mcad.obsm[f'5kCG_u{npc}_tsne'] = mcad.obsm['X_tsne'].copy()


In [ ]:
sc.pp.neighbors(mcad, use_rep=obsm_key, n_neighbors=25)
sc.tl.leiden(mcad, resolution=1.0, key_added=f'5kCG_u{npc}_leiden')

In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)

mcad.obsm[f'X_{coord_base}'] = mcad.obsm[f'5kCG_u{npc}_tsne'].copy()
dump_embedding(mcad, coord_base)
tmp = mcad.obs.copy()
ax = axes[0]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue='Donor',
                        text_anno='Donor',
                        labelsize=8,
                        s=ds,
                        palette='tab10',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        show_legend=True
                        )
ax = axes[1]
_ = categorical_scatter(data=tmp,
                        ax=ax,
                        coord_base=coord_base,
                        hue=f'5kCG_u{npc}_leiden',
                        text_anno=f'5kCG_u{npc}_leiden',
                        labelsize=8,
                        s=ds,
                        palette='tab20',
                        scatter_kws={'rasterized':True},
                        # legend_kws={'ncol':1},
                        # show_legend=True
                        )


In [ ]:
mcad = anndata.AnnData(obs=mcad.obs, obsm=mcad.obsm, obsp=mcad.obsp)
mcad.write_h5ad(f'{outdir}5kCG_embed.h5ad')


In [ ]:
meta = pd.read_csv(f'{indir}clustering/merged/5kCG100k3C_summary.csv.gz', header=0, index_col=0)
meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']] = meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']].astype(str)
meta = meta.loc[meta['Tissue']==tissue]
meta

In [ ]:
mcds = MCDS.open(f'{m3c_dir}*{tissue}*.mcds', use_obs=meta.index, var_dim=var_dim)
mcds = mcds.remove_chromosome(exclude_chromosome=chrom_to_remove, var_dim=var_dim)
mcds

In [ ]:
mcad = mcds.get_score_adata(mc_type='CGN', quant_type='hypo')
binarize_matrix(mcad, cutoff=0.95)
filter_regions(mcad, n_cell=5)
remove_black_list_region(mcad, black_list_path=black_list_path)
mcad.write_h5ad(f'{indir}clustering/tissue/L1/{tissue}/5kCG.h5ad')


In [ ]:
adata1 = anndata.read_h5ad(f'{outdir}5kCG.h5ad')
adata2 = anndata.read_h5ad(f'{indir}clustering/tissue/L1/{tissue}/5kCG.h5ad')
print(adata1.shape, adata2.shape)


In [ ]:
meta = anndata.read_h5ad(f'{outdir}5kCG_embed.h5ad').obs
adata1.obs = meta.loc[adata1.obs.index].copy()
# adata1.obs['Donor'] = adata1.obs['Donor'].replace({'JFINP': 'PT-1LVAN', 'IOBHV': 'PT-1LGRB'})
adata1.obs['Donor'].value_counts()

In [ ]:
meta = pd.read_csv(f'{indir}clustering/merged/5kCG100k3C_summary.csv.gz', header=0, index_col=0)
meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']] = meta[['Donor', 'Tissue', 'cluster', 'subtype', 'majortype']].astype(str)
meta = meta.loc[meta['Tissue']==tissue]
L1_meta = pd.read_csv(f'{indir}L1color.tsv', sep='\t', header=0, index_col=0)
L1_meta = L1_meta.drop(['c35','c36'], axis=0)
L1_color = L1_meta['color'].to_dict()
L1_annot = L1_meta['L1_abbr'].to_dict()
L2_meta = pd.read_csv(f'{indir}subtype_meta.tsv', sep='\t', header=0, index_col=0)
L2_annot = L2_meta['celltype_L2_both_abbr'].to_dict()
adata2.obs = meta.loc[adata2.obs.index].copy()
adata2.obs['L2_annot'] = adata2.obs['subtype'].map(L2_annot).astype(str)
adata2.obs['L1_annot'] = adata2.obs['majortype'].map(L1_annot).astype(str)


In [ ]:
count = adata2.obs['L2_annot'].value_counts()
selct = np.sort(count.index[:20])
l2_palette = {xx:yy for xx,yy in zip(selct, sns.color_palette('tab20'))}
l2_palette[selct[14]] = (0.0, 0.0, 0.0)
l2_palette[selct[15]] = (0.4980392156862745, 0.4980392156862745, 0.4980392156862745)


In [ ]:
count = adata1.obs['5kCG_u16_leiden'].value_counts()
leiden_palette = {xx:yy for xx,yy in zip(count.index, sns.color_palette('tab20', count.shape[0]))}


In [ ]:
for donor in adata1.obs['Donor'].unique():
    tmp1 = adata1[adata1.obs['Donor']==donor].copy()
    tmp2 = adata2[adata2.obs['Donor']==donor].copy()
    tmp1.var['n_cells'] = adata1.X.sum(axis=0).A1
    tmp2.var['n_cells'] = adata2.X.sum(axis=0).A1
    selb = tmp1.var.index[tmp1.var['n_cells']>5].intersection(tmp2.var.index[tmp2.var['n_cells']>5])
    tmp1 = tmp1[:, selb].copy()
    tmp2 = tmp2[:, selb].copy()
    model = LSI(scale_factor=10000,
                n_components=50,
                algorithm='arpack',
                random_state=0)

    model.fit_transform(tmp1, obsm_name='lsi_all')
    model.transform(tmp2, obsm_name='lsi_all')
    npc = significant_pc_test(tmp1, p_cutoff=0.05, obsm='lsi_all', update=False)
    tmp1.obsm['X_lsi'] = normalize(tmp1.obsm['lsi_all'][:, :npc], axis=1)
    tmp2.obsm['X_lsi'] = normalize(tmp2.obsm['lsi_all'][:, :npc], axis=1)
    tmp1.obs['study'] = 'mCT'
    tmp2.obs['study'] = 'm3C'
    adata_list = [tmp1, tmp2]
    integrator = SeuratIntegration()
    integrator.find_anchor(adata_list,
                            k_local=None,
                            key_local='X_lsi',
                            k_anchor=5,
                            key_anchor='X',
                            dim_red='lsi-cca',
                            max_cc_cells=50000,
                            k_score=30,
                            k_filter=None,
                            scale_list=[False, False],
                            alignments=[[[0],[1]]],
                            n_components=npc,
                            n_features=200)
    # thres = np.percentile(anchor['dist_pc'], 60)
    # integrator.anchor[(0,1)] = anchor.loc[anchor['dist_pc']<thres, ['x1','x2','score']].copy()
    corrected = integrator.integrate(key_correct='X_lsi',
                                    row_normalize=True,
                                    n_components=npc,
                                    k_weight=100,
                                    alignments=[[[0],[1]]],
                                    sd=1)
    adata_merge = anndata.AnnData(obs=pd.concat([tmp1.obs, tmp2.obs], axis=0))
    adata_merge.obsm['lsi_all'] = np.concatenate([tmp1.obsm['lsi_all'], tmp2.obsm['lsi_all']], axis=0)
    corrected = pd.DataFrame(normalize(np.concatenate(corrected, axis=0), axis=1), 
                            index=np.concatenate([xx.obs.index for xx in adata_list]))

    adata_merge.obsm[f'5kCG_u{npc}_seuratcc{npc}'] = corrected.loc[adata_merge.obs.index].values

    tsne(adata_merge, obsm=f'5kCG_u{npc}_seuratcc{npc}', metric='euclidean', exaggeration=-1, perplexity=50, n_jobs=-1)
    adata_merge.obsm[f'5kCG_u{npc}_seuratcc{npc}_tsne'] = adata_merge.obsm['X_tsne'].copy()
    adata_merge.write_h5ad(f'{outdir}{tissue}_mCT_m3C_{donor}_merged.h5ad')



In [ ]:
ds = 4
coord_base = 'tsne'

fig, axes = plt.subplots(1, 2, figsize=(10, 4), dpi=300, constrained_layout=True)
for i,donor in enumerate(adata1.obs['Donor'].unique()):
    adata_merge = anndata.read_h5ad(f'{outdir}{tissue}_mCT_m3C_{donor}_merged.h5ad')
    # adata_merge.obsm[f'X_{coord_base}'] = adata_merge.obsm[f'5kCG_u{npc}_seuratcc{npc}_tsne'].copy()
    dump_embedding(adata_merge, coord_base)
    tmp = adata_merge.obs.loc[adata_merge.obs['study']=='mCT'].copy()
    ax = axes[0]
    ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue=f'5kCG_u16_leiden',
                            # text_anno=f'5kCG_u17hm_leiden',
                            labelsize=8,
                            s=ds,
                            palette=leiden_palette,
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1},
                            # show_legend=True,
                            axis_format='empty',
                            )
    for yy,(x,y) in tmp.groupby('5kCG_u16_leiden')[[f'{coord_base}_0', f'{coord_base}_1']].median().iterrows():
        ax.text(x, y, yy, ha='center', va='center')

    tmp = adata_merge.obs.loc[adata_merge.obs['study']=='m3C'].copy()
    ax = axes[1]
    ax.scatter(adata_merge.obsm[f'X_{coord_base}'][:,0], adata_merge.obsm[f'X_{coord_base}'][:,1], s=1, c='lightgrey', alpha=0.5, rasterized=True)
    tmp = tmp.loc[tmp['L2_annot'].isin(selct)].copy()
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue=f'L2_annot',
                            # text_anno=f'L2_annot',
                            labelsize=8,
                            s=ds,
                            palette=l2_palette,
                            scatter_kws={'rasterized':True},
                            legend_kws={'ncol':1},
                            show_legend=True,
                            axis_format='empty',
                            )
    for yy,(x,y) in tmp.groupby('L1_annot')[[f'{coord_base}_0', f'{coord_base}_1']].median().iterrows():
        ax.text(x, y, yy, ha='center', va='center')

for i,mode in enumerate(['snmCT-seq', 'snm3C-seq']):
    axes[i].set_title(mode)

fig.savefig(f'{outdir}{tissue}_mCT_m3C_integration_seurat.pdf', transparent=True)
